In [ ]:
from dataclasses import dataclass
from pathlib import Path

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images
import pandas as pd

import abc_selective_attention_utils as utils


In [ ]:
print(list(kbench.llms.keys()))

In [ ]:
FAMILY = "selective attention"
ATTENTIONAL_BASIS = "structure sensitive"
MODALITY = "visual"
TASK_TYPE = "counting"
TASK_NAME = f"{FAMILY} - {ATTENTIONAL_BASIS} {MODALITY} {TASK_TYPE}"


In [ ]:
@dataclass
class CountAnswer:
    count: int


In [ ]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
TASK_PATH = DATASET_ROOT / "selective_attention/structure_sensitive/visual"
CSV_PATH = TASK_PATH / "structure_sensitive_visual_counting.csv"

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print(TASK_PATH)
print("rows:", len(df))
print("columns:", sorted(df.columns.tolist()))


In [ ]:
base_cols = [
    "scene_id",
    "seed",
    "family",
    "attentional_basis",
    "modality",
    "dimension",
    "variant",
    "principle",
    "condition_type",
    "layout_pattern",
    "difficulty",
    "generator_case",
    "task_type",
    "queried_feature",
    "num_groups",
]

optional_cols = [
    col
    for col in [
        "grouping_feature_primary",
        "grouping_feature_secondary",
        "target_value",
        "planned_target_in_anchor_group",
        "planned_target_outside_anchor_group",
        "planned_non_target_in_anchor_group",
        "actual_target_in_anchor_group",
        "actual_target_outside_anchor_group",
        "actual_non_target_in_anchor_group",
        "min_items_per_group",
        "max_items_per_group",
        "cluster_spread",
        "inter_group_margin",
        "stripe_count",
        "shape_bias_strength",
        "path_crossing_count",
        "region_count",
        "loose_item_count",
        "width",
        "height",
        "margin",
        "min_gap",
        "jitter",
    ]
    if col in df.columns
]

required_cols = ["image_path", "count_prompt", "gold_count"]
missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_required_cols}")

count_eval_df = df[base_cols + optional_cols + required_cols].copy()
groupings = [
    ["dimension"],
    ["dimension", "variant"],
    ["principle"],
    ["principle", "condition_type"],
    ["principle", "layout_pattern"],
]

failure_cols = [
    "scene_id",
    "seed",
    "dimension",
    "variant",
    "generator_case",
    "gold_count",
    "predicted_count",
    "failure_type",
]

print("count_eval_df columns:", count_eval_df.columns.tolist())
print("groupings:", groupings)


In [ ]:
@kbench.task(store_task=False)
def single_structure_sensitive_visual_count(
    llm,
    scene_id,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    principle,
    condition_type,
    layout_pattern,
    difficulty,
    generator_case,
    task_type,
    queried_feature,
    num_groups,
    image_path,
    count_prompt,
    gold_count,
    grouping_feature_primary=None,
    grouping_feature_secondary=None,
    target_value=None,
    planned_target_in_anchor_group=None,
    planned_target_outside_anchor_group=None,
    planned_non_target_in_anchor_group=None,
    actual_target_in_anchor_group=None,
    actual_target_outside_anchor_group=None,
    actual_non_target_in_anchor_group=None,
    min_items_per_group=None,
    max_items_per_group=None,
    cluster_spread=None,
    inter_group_margin=None,
    stripe_count=None,
    shape_bias_strength=None,
    path_crossing_count=None,
    region_count=None,
    loose_item_count=None,
    width=None,
    height=None,
    margin=None,
    min_gap=None,
    jitter=None,
) -> dict:
    gold = int(gold_count)

    pred = None
    error = None
    error_type = None

    try:
        resolved_image_path = TASK_PATH / image_path
        img = images.from_path(resolved_image_path)
        response = llm.prompt(count_prompt, image=img, schema=CountAnswer)
        pred = int(response.count)
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = "ok" if is_correct else (error_type if has_error else "wrong_answer")

    kbench.assertions.assert_true(
        is_correct,
        expectation=f"Expected {gold}, got {pred}" + (f" | error_type={error_type} | error={error}" if error else ""),
    )

    result = {
        "task": "structure_sensitive_visual_count_v1",
        "model": llm.name,
        "scene_id": scene_id,
        "seed": int(seed),
        "family": family,
        "attentional_basis": attentional_basis,
        "modality": modality,
        "dimension": dimension,
        "variant": variant,
        "principle": principle,
        "condition_type": condition_type,
        "layout_pattern": layout_pattern,
        "difficulty": difficulty,
        "generator_case": generator_case,
        "task_type": task_type,
        "queried_feature": queried_feature,
        "num_groups": int(num_groups),
        "gold_count": gold,
        "predicted_count": pred,
        "image_path": str(image_path),
        "is_correct": is_correct,
        "is_failure": is_failure,
        "has_error": has_error,
        "failure_type": failure_type,
    }

    if error_type is not None:
        result["error_type"] = error_type
    if error is not None:
        result["error"] = error
    return result


@kbench.task(name="selective attention - structure sensitive visual counting")
def structure_sensitive_visual_count_dataset(llm, df) -> tuple[float, float]:
    with kbench.client.enable_cache():
        runs = single_structure_sensitive_visual_count.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            evaluation_data=df,
            n_jobs=5,
            timeout=900,
            remove_run_files=False,
        )

    result = utils.summarize_and_log_runs(TASK_NAME, llm.name, runs, groupings, failure_cols)
    return result


In [ ]:
run = structure_sensitive_visual_count_dataset.run(kbench.llm, count_eval_df)
print(run)


In [ ]:
%choose structure_sensitive_visual_count_dataset